# Module 03 — Control flow

`if`, loops, `match`. Little of this is new; the sections below are the places
where Python does not do what C or Java would.

Predictions are `assert` with `...` for your answer. Silence means right.

## 1. Blocks are indentation

No braces, no parentheses around the condition, and a colon opens the block. Four
spaces by convention — and it is not cosmetic: the indentation *is* the syntax, so
there is no dangling-else problem and no way for the layout to lie about the
structure.

In [ ]:
reading = 91.0

if reading > 85:
    print("above limit")
    print("still in the same block")
elif reading < -40:
    print("below range")
else:
    print("plausible")

`elif` is one word. There is no `switch` — `match` further down is something else.

## 2. The conditional expression

Python's answer to `?:` puts the condition in the middle:

```python
state = "high" if reading > 85 else "ok"
```

Same evaluation rule as `?:` — only the branch that is taken gets evaluated.

In [ ]:
reading = 91.0
print("high" if reading > 85 else "ok")

## 3. `for` is for-each, and nothing else

There is no `for (int i = 0; i < n; i++)`. A Python `for` walks a sequence, and
counting is something you ask for explicitly:

- `range(start, stop, step)` — the numbers, `stop` exclusive as you would expect
- `enumerate(seq)` — index and item together
- `zip(a, b)` — two sequences in step, stopping at the shorter one

`range` is lazy: it is an object that yields numbers, not a list of them.

In [ ]:
print(range(3), type(range(3)))
print(list(range(3)), list(range(2, 8, 2)))

readings = [21.7, 23.1, 91.0]
for i, value in enumerate(readings, start=1):
    print(i, value)

times = ["14:05", "14:10"]
for t, value in zip(times, readings):
    print(t, value)

**Predict.** `range(0, 10, 3)` — how many values, and what is the last one?

In [ ]:
assert len(range(0, 10, 3)) == ...
assert list(range(0, 10, 3))[-1] == ...

The idiom to unlearn: `for i in range(len(readings))` and then `readings[i]`. If
you want the item, iterate the sequence; if you want both, use `enumerate`.

## 4. The loop variable outlives the loop

Python has no block scope. A name bound inside `for`, `while` or `if` is still
bound after it — the enclosing function is the scope, not the braces.

In [ ]:
for i in range(3):
    pass

# What is i now?
assert i == ...

In C and Java `i` is scoped to the loop and gone afterwards. Here it survives —
with a catch: the name is bound only if the body ran at least once. After
`for i in []: pass` there is no `i` at all, and reading it raises `NameError`.

So code that uses the loop variable afterwards works until the sequence is empty
once.

## 5. `else` on a loop

This one has no counterpart in any language you are likely to know: a loop can
carry an `else`, and it runs **when the loop finished without `break`**.

Read it as "no break" rather than "else" — it is the search idiom, and it removes
the found-flag you would otherwise carry.

In [ ]:
readings = [21.7, 23.1, 22.4]

for value in readings:
    if value > 85:
        print("first implausible:", value)
        break
else:
    print("all readings plausible")

**Predict** what the loop below prints — one line, or two, or none:

In [ ]:
found = []

for value in [21.7, 91.0, 22.4]:
    if value > 85:
        found.append(value)
        break
else:
    found.append("none")

assert found == ...

## 6. `while`, `break`, `continue`

As you know them. There is no `do ... while`; the usual stand-in is
`while True:` with a `break` at the point where the condition is decided.

In [ ]:
countdown = 3
while countdown:  # any non-zero number is truthy
    print(countdown)
    countdown -= 1

for value in [21.7, None, 23.1]:
    if value is None:
        continue  # skip the gap in the log
    print(value)

## 7. `match` is not `switch`

`match` (3.10+) tests **shapes**, not just values, and binds parts of what it
matched. There is no fallthrough and therefore no `break`; the first matching
case wins.

In [ ]:
def describe(status):
    match status:
        case 0:
            return "idle"
        case 1 | 2:
            return "warming up"  # or, not fallthrough
        case int() as code if code > 100:
            return f"fault {code}"  # guard, and the value is bound
        case _:
            return "unknown"


for s in (0, 2, 250, 7):
    print(s, "->", describe(s))

`case _` is the default. And a bare name in a pattern **binds** rather than
compares — which is the one rule here worth seeing fail. Run this:

For a plain value switch, `if`/`elif` is shorter and clearer. `match` earns its
place when you are taking apart a structure — a tuple, a dict, an object — which
is module 06 and later.

In [ ]:
# Written as text and handed to compile(), because a cell that cannot be parsed
# would take the whole notebook with it. compile() reaches the same verdict.
broken = """
match code:
    case IDLE:      # meant as: code == IDLE. Actually: binds IDLE to whatever code is.
        print("idle")
    case 5:
        print("five")
"""

compile(broken, "<demo>", "exec")

```
SyntaxError: name capture 'IDLE' makes remaining patterns unreachable
```

`case IDLE:` matched everything and rebound `IDLE`, so the `case 5:` below it can
never run — and Python refuses to compile that rather than letting it through. As
the *last* case it does compile, and is then a `case _` wearing a name.

Constants in patterns therefore have to be dotted — `Status.IDLE`, `Colour.RED` —
or written as literals.

---

On to `exercises/`, then `uv run pytest 03_control_flow`.